In [74]:
import matplotlib.pyplot as plt
import numpy as np
import requests
import seaborn as sns
import torch
from pathlib import Path
import os
from io import BytesIO
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from transformers import CLIPProcessor, CLIPModel
import re

In [75]:
#Remove any files with below 50 images from use for training
DIR_PATH = Path.cwd() / 'data' / 'GBIF_Pre_Processed'
LESS_THAN_PATH = Path.cwd() / 'data' / 'GBIF_UNDER_50'
image_dist = []
for item in DIR_PATH.iterdir():

    if len(list(Path(item).iterdir())) < 50:
        item.rmdir()


In [76]:
# Look through image files and collect any that have below 100, store the plant symbol and the amount of images it has

#Get PATH
DIR_PATH = Path.cwd() / 'data' / 'GBIF_Pre_Processed'

image_dist = []
for item in DIR_PATH.iterdir():

    symbol_and_count = (item.name,len(list(Path(item).iterdir())))
    image_dist.append(symbol_and_count)


less_than_100 = [x for x in image_dist if x[1] < 100]

print(f"There are {len(less_than_100)} plant folders within the dataset that have less than 100 images")




There are 359 plant folders within the dataset that have less than 100 images


In [77]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [78]:
def get_image_embeddings(model, processor, images):
    inputs = processor(images,return_tensors="pt", padding=True)
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)

        embeddings = image_features.pooler_output

        image_features = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
    #image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    return image_features #.numpy()

In [79]:
def decompose_embeddings_pca(embeddings, n_components = 2):
    pca = PCA(n_components=n_components)
    reduced_embeddings = pca.fit_transform(embeddings)
    return reduced_embeddings, pca

In [80]:
def cluster_embeddings(embeddings, n_clusters=2):
    kmeans = KMeans(n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)
    return labels, kmeans

In [81]:
def visualize_results(reduced_emb, labels, images, title="GBIF Plant image analysis"):
    plt.figure(figsize=(12,10))
    ax = plt.gca()

    unique_labels = np.unique(labels)
    palette = sns.color_palette("viridis", len(unique_labels))

    #plt.scatter(reduced_emb[:,0], reduced_emb[:, 1], cmap='viridis', c=labels)

    #IF YOU WANT TO PLOT IMAGES UN COMMENT AND COMMENT OUT SCATTER ABOVE
    for i, (x,y) in enumerate(reduced_emb):

        img = images[i].copy()
        img.thumbnail((100,100))

        imagebox = OffsetImage(img, zoom=0.3)

        ab = AnnotationBbox(imagebox, (x,y),
                            xycoords='data',
                            boxcoords='offset points',
                            pad = 0.3,
                            bboxprops=dict(edgecolor=palette[labels[i]], linewidth=2))
        ax.add_artist(ab)

    x_min, x_max = reduced_emb[:, 0].min(), reduced_emb[:,0].max()
    y_min, y_max = reduced_emb[:, 1].min(), reduced_emb[:,1].max()

    margin_x = (x_max - x_min) * 0.2
    margin_y = (y_max - y_min) * 0.2
    plt.xlim(x_min - margin_x, x_max + margin_x)
    plt.ylim(y_min - margin_y, y_max + margin_y)

    plt.title(title)
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.grid(True, alpha=0.3)

    output_file = f".\data\GBIF_IMAGE_EXPLORATION\{title}.png"
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
#Load in images

#Get PATH
DIR_PATH = Path.cwd() / 'data' / 'GBIF_Pre_Processed'

img_path = None
counter = 0
for item in DIR_PATH.iterdir():
    img_path = item

    images = []
    for path in img_path.iterdir():
        images.append(Image.open(path).convert("RGB"))

    embeddings = get_image_embeddings(model, processor, images)

    reduced_emb, pca = decompose_embeddings_pca(embeddings=embeddings, n_components=2)

    labels, kmeans = cluster_embeddings(embeddings=embeddings, n_clusters=2)

    visualize_results(reduced_emb=reduced_emb, labels=labels, images=images, title=f"{item.name}_Embedding_Analysis")

